In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scanpy as sc
import squidpy as sq
import alphashape

from shapely.geometry import Point, Polygon
from shapely.ops import unary_union
from matplotlib.patches import Circle, Rectangle
from IPython.display import display

## Paths & data loading

In [ ]:
# Set path to input AnnData object
adata = sc.read_h5ad("")

In [ ]:
# ============================================
# Immune infiltration per KO polygon + mixing
# using the same `adata` as the KO island builder
# ============================================
import numpy as np
import pandas as pd
from shapely import wkt as shapely_wkt
from shapely.geometry import Point
from shapely.strtree import STRtree

# --- your AnnData with islands already stored ---
A = adata   # same as in the island builder

# -----------------------------
# basic safety checks
# -----------------------------
assert "ko_islands" in A.uns and isinstance(A.uns["ko_islands"], dict), \
    "No islands found in A.uns['ko_islands'] – run the KO island builder first."

assert "Phenotype" in A.obs.columns, "'Phenotype' not found in A.obs"

# Per-cell annotations
coords = np.asarray(A.obsm["spatial"])
n_cells = coords.shape[0]

# We'll treat Phenotype as the per-cell 'subtype'
subtg = A.obs["Phenotype"].astype("string")

# center/periphery labels from the island builder
tumor_region = A.obs.get(
    "tumor_region",
    pd.Series(index=A.obs.index, dtype="string")
).astype("string")

# If you want an explicit KO label per cell (should match what purity used)
ko_cell_norm = subtg.str.strip().str.lower()

# Pixel size (if not still in memory)
try:
    UM_PER_PIXEL
except NameError:
    UM_PER_PIXEL = 0.5  # adjust if needed

# -----------------------------
# 1) Rehydrate polygons
# -----------------------------
is_tab = A.uns["ko_islands"]

poly_ids     = list(is_tab["poly_id"])
kos_raw      = list(is_tab["ko"])
purities     = list(is_tab["purity"])
areas_px2    = list(is_tab["area_px2"])
areas_um2    = list(is_tab["area_um2"])
polys_wkt    = list(is_tab["polygon_wkt"])
libs         = list(is_tab.get("library_id", ["NA"] * len(poly_ids)))

polys = [shapely_wkt.loads(w) if isinstance(w, str) and len(w) else None
         for w in polys_wkt]

# Filter out missing/empty polygons
valid_mask = [(p is not None and (not p.is_empty)) for p in polys]
poly_ids   = [pid for pid, ok in zip(poly_ids, valid_mask) if ok]
kos_raw    = [ko  for ko,  ok in zip(kos_raw,   valid_mask) if ok]
purities   = [pu  for pu,  ok in zip(purities,  valid_mask) if ok]
areas_px2  = [ar  for ar,  ok in zip(areas_px2, valid_mask) if ok]
areas_um2  = [ar  for ar,  ok in zip(areas_um2, valid_mask) if ok]
libs       = [lb  for lb,  ok in zip(libs,      valid_mask) if ok]
polys      = [p   for p,   ok in zip(polys,     valid_mask) if ok]

# normalized KO label per polygon
kos_norm = [str(ko).strip().lower() for ko in kos_raw]

# -----------------------------
# 2) Build a point STRtree
# -----------------------------
points_all = [Point(xy) for xy in coords]
pt_tree    = STRtree(points_all)
geom_to_idx = {id(g): i for i, g in enumerate(points_all)}

def _hits_to_indices(hits):
    # Shapely 2.x returns geometry objects; Shapely 1.x may return ints
    if isinstance(hits, np.ndarray):
        return hits.tolist()
    if not hits:
        return []
    first = hits[0]
    if isinstance(first, (int, np.integer)):  # Shapely 1.x
        return list(hits)
    return [geom_to_idx[id(g)] for g in hits]

# -----------------------------
# 3) Define "immune" and subtype levels
# -----------------------------
# We don't know your exact labels, so we keep this generic.
# You can later define which Phenotype values are immune in Python/R.
all_subtypes = pd.Series(subtg).fillna("NA").astype(str)
subtype_levels = sorted(all_subtypes.unique().tolist())

# If you *do* have a known set of immune phenotypes, define them here:
IMMUNE_LABELS = set()  # e.g. {"CD8_Tcell", "Treg", "Macrophage"}  <-- fill in if you want
subtg_str = all_subtypes.to_numpy()

# Polygon is called "mixed" below this purity
MIXED_PURITY_THRESHOLD = 0.7

# ------------------------------------------
# 4) Per-polygon membership + infiltration + mixing
# ------------------------------------------
records = []

for pid, ko_raw, ko_norm, pur, area_px2, area_um2, lib, poly in zip(
        poly_ids, kos_raw, kos_norm, purities, areas_px2, areas_um2, libs, polys):

    # candidate points via spatial index
    hits = pt_tree.query(poly)
    idxs = _hits_to_indices(hits)

    if len(idxs) == 0:
        records.append({
            "poly_id": pid,
            "library_id": lib,
            "ko": ko_raw,
            "ko_norm": ko_norm,
            "ko_purity": float(pur),
            "ko_mixing_frac": float(1.0 - pur),
            "ko_is_mixed": bool(pur < MIXED_PURITY_THRESHOLD),
            "area_um2": float(area_um2),
            "area_mm2": float(area_um2) / 1e6,
            "n_cells_total": 0,
            "n_immune_total": 0,
            "immune_frac": np.nan,
            "immune_density_per_mm2": np.nan,
            "n_center": 0,
            "n_periphery": 0,
        })
        continue

    # keep only true inside / on-boundary points
    inside = []
    for j in idxs:
        if poly.covers(points_all[j]):
            inside.append(j)

    if not inside:
        records.append({
            "poly_id": pid,
            "library_id": lib,
            "ko": ko_raw,
            "ko_norm": ko_norm,
            "ko_purity": float(pur),
            "ko_mixing_frac": float(1.0 - pur),
            "ko_is_mixed": bool(pur < MIXED_PURITY_THRESHOLD),
            "area_um2": float(area_um2),
            "area_mm2": float(area_um2) / 1e6,
            "n_cells_total": 0,
            "n_immune_total": 0,
            "immune_frac": np.nan,
            "immune_density_per_mm2": np.nan,
            "n_center": 0,
            "n_periphery": 0,
        })
        continue

    inside = np.array(inside, dtype=int)
    n_total = inside.size

    # center / periphery split
    tr_vals = tumor_region.iloc[inside].fillna("Other").astype(str)
    n_center    = int((tr_vals == "center").sum())
    n_periphery = int((tr_vals == "periphery").sum())

    # immune vs non-immune (if IMMUNE_LABELS is defined)
    st_vals = subtg.iloc[inside].fillna("NA").astype(str).to_numpy()
    if IMMUNE_LABELS:
        immune_mask_inside = np.isin(st_vals, list(IMMUNE_LABELS))
    else:
        immune_mask_inside = np.zeros_like(st_vals, dtype=bool)  # all non-immune by default

    n_immune = int(immune_mask_inside.sum())
    immune_frac = (n_immune / n_total) if n_total > 0 else np.nan
    immune_density_per_mm2 = (
        n_immune / (area_um2 / 1e6)
        if (area_um2 is not None and area_um2 > 0)
        else np.nan
    )

    # detailed per-Phenotype counts + fractions
    st_counts = (
        pd.Series(st_vals, dtype="string")
          .value_counts(dropna=False)
          .reindex(subtype_levels, fill_value=0)
    )

    row = {
        "poly_id": pid,
        "library_id": lib,
        "ko": ko_raw,
        "ko_norm": ko_norm,

        # *** mixing information ***
        "ko_purity": float(pur),
        "ko_mixing_frac": float(1.0 - pur),
        "ko_is_mixed": bool(pur < MIXED_PURITY_THRESHOLD),

        # geometry / size
        "area_um2": float(area_um2),
        "area_mm2": float(area_um2) / 1e6,

        # counts
        "n_cells_total": int(n_total),
        "n_immune_total": int(n_immune),
        "immune_frac": float(immune_frac),
        "immune_density_per_mm2": float(immune_density_per_mm2),

        # center vs periphery
        "n_center": int(n_center),
        "n_periphery": int(n_periphery),
    }

    # wide columns for each Phenotype count and fraction
    for st in subtype_levels:
        cnt = int(st_counts.get(st, 0))
        row[f"Phenotype__{st}__count"] = cnt
        row[f"Phenotype__{st}__frac"]  = (cnt / n_total) if n_total > 0 else np.nan

    records.append(row)

infiltration_df = (
    pd.DataFrame.from_records(records)
      .sort_values(["ko", "library_id", "poly_id"])
      .reset_index(drop=True)
)

# -----------------------------
# 5) Store in A.uns
# -----------------------------
A.uns["ko_islands_infiltration"] = {
    "phenotype_levels": subtype_levels,
    "mixed_threshold": float(MIXED_PURITY_THRESHOLD),
    "table": infiltration_df.to_dict(orient="list"),
}

# Per-KO rollup (as before)
infiltration_per_ko = (
    infiltration_df
      .groupby("ko", as_index=False)
      .agg({
          "area_mm2": "sum",
          "n_cells_total": "sum",
          "n_immune_total": "sum"
      })
)
infiltration_per_ko["immune_frac_weighted"] = (
    infiltration_per_ko["n_immune_total"] / infiltration_per_ko["n_cells_total"]
).where(infiltration_per_ko["n_cells_total"] > 0, np.nan)
infiltration_per_ko["immune_density_per_mm2"] = (
    infiltration_per_ko["n_immune_total"] / infiltration_per_ko["area_mm2"]
).where(infiltration_per_ko["area_mm2"] > 0, np.nan)

center_periph = (
    infiltration_df
      .loc[:, ["poly_id", "ko", "n_center", "n_periphery", "n_cells_total"]]
      .assign(center_frac=lambda d: d["n_center"] / d["n_cells_total"])
      .assign(periph_frac=lambda d: d["n_periphery"] / d["n_cells_total"])
)

A.uns["ko_islands_infiltration_summary"] = {
    "per_polygon": infiltration_df.to_dict(orient="list"),
    "per_ko": infiltration_per_ko.to_dict(orient="list"),
    "center_periphery": center_periph.to_dict(orient="list"),
}

print(infiltration_df.head(3))
print(infiltration_per_ko.sort_values("immune_density_per_mm2", ascending=False).head(10))


In [ ]:
# Recover the infiltration table
inf_df = pd.DataFrame(A.uns["ko_islands_infiltration"]["table"])

# Export to CSV
inf_df.to_csv("../analysis/TME_polygons/ko_islands_infiltration_SE1_F8.csv", index=False)

print("Saved as ko_islands_infiltration.csv")


## TME analysis - ROI subsampling

#### define ROIs

In [ ]:

# ============================================
# CONFIG
# ============================================
KO_COL        = "Phenotype"
KO_WHITELIST  = {"F8", "Serpine1"}
MOUSE_COL     = "library_id"
SPATIAL_KEY   = "spatial"

ROI_RADIUS    = 200
MIN_CELLS_ROI = 400
MIN_KO_CELLS  = 200

MIX_THRESHOLD = 0.20

# target numbers per group
N_LOW_F8    = 20   # low-mixing, majority F8
N_LOW_SE1   = 20   # low-mixing, majority Serpine1
N_HIGH_MIX  = 20   # high-mixing, any majority

# tumor border parameters (in pixel units)
ALPHA_TUMOR         = 0.04   # alpha-shape parameter for tumor polygon
TUMOR_BORDER_WIDTH  = 150.0  # thickness (px) of border band to exclude
MIN_TUMOR_AREA_FRAC = 0.05   # keep lobes with ≥5% of max tumor area

rng = np.random.default_rng(42)

A = adata

coords = np.asarray(A.obsm[SPATIAL_KEY])
phenos = A.obs[KO_COL].astype(str).values
mice   = A.obs[MOUSE_COL].astype(str).values

phenotype_levels = sorted(np.unique(phenos))
ROI_AREA = float(np.pi * ROI_RADIUS**2)
ROI_RADIUS_SQ = ROI_RADIUS ** 2

# --------------------------------------------
# Optional manual exclusion boxes
# --------------------------------------------
EXCLUDE_BOXES = {
    # "CF497": [
    #     (2000.0, 6000.0, 5000.0, 9000.0),
    # ]
}


def build_exclude_mask(coords_mouse, mouse_id):
    """
    Returns a boolean mask (True = exclude) for coords_mouse
    based on EXCLUDE_BOXES.
    """
    boxes = EXCLUDE_BOXES.get(mouse_id)
    if not boxes:
        return np.zeros(coords_mouse.shape[0], dtype=bool)

    x = coords_mouse[:, 0]
    y = coords_mouse[:, 1]
    mask = np.zeros(coords_mouse.shape[0], dtype=bool)

    for (xmin, xmax, ymin, ymax) in boxes:
        in_box = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
        mask |= in_box

    return mask


# --------------------------------------------
# Tumor geometry from F8 + Serpine1 cells
# --------------------------------------------
def build_tumor_polygon_for_mouse(coords_mouse, phenos_mouse,
                                  alpha=ALPHA_TUMOR):
    """
    Build tumor geometry as union of alpha-shapes of F8 + Serpine1 cells.
    Can return Polygon or MultiPolygon (multiple tumor lobes) or None.
    """
    tumor_mask = np.isin(phenos_mouse, ["F8", "Serpine1"])
    pts = coords_mouse[tumor_mask]

    if pts.shape[0] < 20:
        return None

    try:
        shape = alphashape.alphashape(pts, alpha)
    except Exception:
        return None

    if shape is None or shape.is_empty:
        return None

    # Collect polygons
    if shape.geom_type == "Polygon":
        polys = [shape]
    else:
        polys = [
            g for g in getattr(shape, "geoms", [])
            if isinstance(g, Polygon) and not g.is_empty and g.area > 0
        ]

    if not polys:
        return None

    # Keep polygons with area >= MIN_TUMOR_AREA_FRAC * max_area
    areas = [p.area for p in polys]
    max_area = max(areas)
    area_cut = max_area * MIN_TUMOR_AREA_FRAC
    big_polys = [p for p, a in zip(polys, areas) if a >= area_cut]

    if not big_polys:
        return None

    # Union them – might still be MultiPolygon if disjoint
    poly = unary_union(big_polys)
    poly = poly.buffer(0)
    if poly.is_empty:
        return None

    return poly


def compute_tumor_masks_for_mouse(coords_mouse, phenos_mouse,
                                  alpha=ALPHA_TUMOR,
                                  border_width=TUMOR_BORDER_WIDTH):
    """
    For all cells of a mouse, compute:
      - inside_mask: inside any tumor lobe (F8+Serpine1 alpha-shapes)
      - border_mask: subset of inside_mask within 'border_width'
                     of nearest tumor lobe exterior.
    Supports Polygon or MultiPolygon. If no tumor polygon, both
    masks are all False.
    """
    n = coords_mouse.shape[0]
    if n == 0:
        return np.zeros(0, dtype=bool), np.zeros(0, dtype=bool)

    poly = build_tumor_polygon_for_mouse(coords_mouse, phenos_mouse, alpha=alpha)
    if poly is None or poly.is_empty:
        return np.zeros(n, dtype=bool), np.zeros(n, dtype=bool)

    # list of lobes
    if poly.geom_type == "Polygon":
        polys = [poly]
    else:
        polys = [
            g for g in getattr(poly, "geoms", [])
            if isinstance(g, Polygon) and not g.is_empty
        ]
        if not polys:
            return np.zeros(n, dtype=bool), np.zeros(n, dtype=bool)

    inside_mask = np.zeros(n, dtype=bool)
    border_mask = np.zeros(n, dtype=bool)

    for i, (x, y) in enumerate(coords_mouse):
        pt = Point(x, y)

        inside_any = False
        min_d = np.inf

        for pg in polys:
            if pg.contains(pt) or pg.touches(pt):
                inside_any = True
                d = pt.distance(pg.exterior)
            else:
                d = pt.distance(pg.exterior)

            if d < min_d:
                min_d = d

        if not inside_any:
            continue

        inside_mask[i] = True
        if min_d < border_width:
            border_mask[i] = True

    return inside_mask, border_mask


# --------------------------------------------
# Visualise tumor polygon(s) for QC (optional)
# --------------------------------------------
def plot_tumor_polygon_for_mouse(mouse_id,
                                 coords, phenos, mice,
                                 show_border_band=True):
    """
    Visualise F8+Serpine1 tumor polygon(s) and border band for a mouse.
    """
    mask_mouse = (mice == mouse_id)
    coords_mouse = coords[mask_mouse, :]
    phenos_mouse = phenos[mask_mouse]

    if coords_mouse.shape[0] == 0:
        print(f"[WARN] No cells for mouse {mouse_id}")
        return

    poly = build_tumor_polygon_for_mouse(coords_mouse, phenos_mouse,
                                         alpha=ALPHA_TUMOR)
    if poly is None or poly.is_empty:
        print(f"[WARN] No tumor polygon for mouse {mouse_id}")
        return

    inside_mask, border_mask = compute_tumor_masks_for_mouse(
        coords_mouse, phenos_mouse,
        alpha=ALPHA_TUMOR,
        border_width=TUMOR_BORDER_WIDTH
    )

    fig, ax = plt.subplots(figsize=(6, 6))

    # all cells
    ax.scatter(coords_mouse[:, 0], coords_mouse[:, 1],
               s=1, alpha=0.1, color="lightgrey", label="all cells")

    # interior (non-border)
    interior_mask = inside_mask & ~border_mask
    ax.scatter(coords_mouse[interior_mask, 0],
               coords_mouse[interior_mask, 1],
               s=2, alpha=0.5, color="grey", label="tumor interior")

    # border band
    if show_border_band:
        ax.scatter(coords_mouse[border_mask, 0],
                   coords_mouse[border_mask, 1],
                   s=3, alpha=0.7, color="red", label="tumor border band")

    # draw all lobes
    if poly.geom_type == "Polygon":
        polys_to_plot = [poly]
    else:
        polys_to_plot = [g for g in poly.geoms if isinstance(g, Polygon)]

    for pg in polys_to_plot:
        x_poly, y_poly = pg.exterior.xy
        ax.plot(x_poly, y_poly, linewidth=1.5, color="black", label="tumor polygon")

        # inner border
        try:
            inner_poly = pg.buffer(-TUMOR_BORDER_WIDTH)
            if not inner_poly.is_empty and inner_poly.geom_type == "Polygon":
                x_inner, y_inner = inner_poly.exterior.xy
                ax.plot(x_inner, y_inner, linewidth=1.0,
                        color="black", linestyle="--")
        except Exception:
            pass

    ax.set_title(f"Tumor polygon(s) for {mouse_id}")
    ax.set_aspect("equal")
    ax.legend(markerscale=5)
    # ax.invert_yaxis()
    plt.show()


# --------------------------------------------
# ROI builder for a single center
# --------------------------------------------
def compute_roi_for_center_local(center_idx_local, coords_mouse, phenos_mouse):
    """
    center_idx_local: index into coords_mouse / phenos_mouse
    coords_mouse: (N_mouse_cells, 2)
    phenos_mouse: (N_mouse_cells,)
    """
    cx, cy = coords_mouse[center_idx_local]

    dx = coords_mouse[:, 0] - cx
    dy = coords_mouse[:, 1] - cy
    dist2 = dx * dx + dy * dy

    inside_mask = dist2 <= ROI_RADIUS_SQ
    if not np.any(inside_mask):
        return None

    roi_indices   = np.where(inside_mask)[0]
    n_cells_total = roi_indices.size
    if n_cells_total < MIN_CELLS_ROI:
        return None

    roi_phenos = phenos_mouse[roi_indices]

    # KO composition
    is_ko = np.isin(roi_phenos, list(KO_WHITELIST))
    ko_phenos = roi_phenos[is_ko]
    n_ko_total = ko_phenos.size
    if n_ko_total < MIN_KO_CELLS:
        return None

    n_F8       = np.sum(ko_phenos == "F8")
    n_Serpine1 = np.sum(ko_phenos == "Serpine1")

    if n_F8 == 0 and n_Serpine1 == 0:
        return None

    if n_F8 >= n_Serpine1:
        majority_ko    = "F8"
        other_ko_count = n_Serpine1
    else:
        majority_ko    = "Serpine1"
        other_ko_count = n_F8

    other_ko_frac = other_ko_count / float(n_ko_total)
    mixing_group  = "low_mixing" if other_ko_frac <= MIX_THRESHOLD else "high_mixing"

    # Phenotype composition
    pheno_counts = {}
    for p in phenotype_levels:
        c = np.sum(roi_phenos == p)
        pheno_counts[p] = c

    pheno_fracs = {f"{p}__frac": (c / n_cells_total) for p, c in pheno_counts.items()}
    pheno_counts = {f"{p}__count": c for p, c in pheno_counts.items()}

    row = {
        "center_x": float(cx),
        "center_y": float(cy),
        "roi_radius": float(ROI_RADIUS),
        "roi_area_units2": ROI_AREA,
        "n_cells_total": int(n_cells_total),
        "n_ko_total": int(n_ko_total),
        "n_F8": int(n_F8),
        "n_Serpine1": int(n_Serpine1),
        "majority_ko": majority_ko,
        "other_ko_frac": float(other_ko_frac),
        "mixing_group": mixing_group,
    }
    row.update(pheno_counts)
    row.update(pheno_fracs)
    return row


# --------------------------------------------
# Farthest-point sampling helper
# --------------------------------------------
def farthest_point_sampling(df, n, rng):
    """
    Select up to n rows from df such that their (center_x, center_y)
    are as far apart as possible (greedy farthest-point sampling).
    """
    if df.shape[0] <= n:
        return df.copy()

    coords_xy = df[["center_x", "center_y"]].to_numpy()
    chosen_idx = []

    # start from a random candidate
    idx0 = int(rng.integers(0, coords_xy.shape[0]))
    chosen_idx.append(idx0)

    while len(chosen_idx) < n:
        chosen_coords = coords_xy[chosen_idx]

        diff = coords_xy[:, None, :] - chosen_coords[None, :, :]
        dist2 = np.sum(diff**2, axis=2)
        min_dist2 = dist2.min(axis=1)

        # don't reselect already chosen
        min_dist2[chosen_idx] = -1.0

        next_idx = int(np.argmax(min_dist2))
        if min_dist2[next_idx] < 0:
            break

        chosen_idx.append(next_idx)

    return df.iloc[chosen_idx].copy()


# ============================================
# Sample ROIs per mouse
# ============================================
records = []

for mouse_id in sorted(np.unique(mice)):
    idx_mouse = np.where(mice == mouse_id)[0]

    coords_mouse = coords[idx_mouse, :]
    phenos_mouse = phenos[idx_mouse]

    if coords_mouse.shape[0] == 0:
        print(f"[WARN] mouse {mouse_id}: no cells, skipping.")
        continue

    # 1) tumor-based masks (multi-lobe aware)
    inside_tumor_mask, border_tumor_mask = compute_tumor_masks_for_mouse(
        coords_mouse, phenos_mouse,
        alpha=ALPHA_TUMOR,
        border_width=TUMOR_BORDER_WIDTH
    )

    # require centers to be inside tumor and away from tumor border
    tumor_interior_mask = inside_tumor_mask & (~border_tumor_mask)

    # 2) manual exclusion boxes (optional)
    box_mask = build_exclude_mask(coords_mouse, mouse_id)

    # final exclude mask
    exclude_mask = (~tumor_interior_mask) | box_mask

    n_excluded = int(exclude_mask.sum())
    candidate_indices_local = np.where(~exclude_mask)[0]

    if candidate_indices_local.size == 0:
        print(f"[WARN] mouse {mouse_id}: no valid candidate centers after tumor filter.")
        continue

    print(
        f"Building candidate ROIs for mouse {mouse_id} "
        f"(candidate centers: {candidate_indices_local.size}, "
        f"excluded cells: {n_excluded})"
    )

    # 3) build full candidate ROI table for this mouse
    candidate_rows = []

    for ii, center_idx_local in enumerate(candidate_indices_local):
        if ii % 5000 == 0 and ii > 0:
            print(
                f"  mouse {mouse_id}: checked {ii} centers, "
                f"{len(candidate_rows)} valid ROIs so far"
            )

        roi_row = compute_roi_for_center_local(
            center_idx_local, coords_mouse, phenos_mouse
        )
        if roi_row is None:
            continue

        roi_row["mouse"] = mouse_id
        roi_row["center_cell_index"] = int(idx_mouse[center_idx_local])
        candidate_rows.append(roi_row)

    if not candidate_rows:
        print(f"  -> mouse {mouse_id}: no valid ROIs after filtering.")
        continue

    df_candidates = pd.DataFrame(candidate_rows)
    print(f"  -> mouse {mouse_id}: {df_candidates.shape[0]} candidate ROIs")

    # 4) farthest-point sample in 3 groups:
    #    - low_F8
    #    - low_Serpine1
    #    - high_mixing (any majority)
    group_specs = [
        ("low_F8",        "low_mixing",  "F8",        N_LOW_F8),
        ("low_Serpine1",  "low_mixing",  "Serpine1",  N_LOW_SE1),
        ("high_mixing",   "high_mixing", None,        N_HIGH_MIX),
    ]

    for label, mix_group, maj_ko, target_n in group_specs:
        if maj_ko is None:
            df_group = df_candidates.query("mixing_group == @mix_group")
        else:
            df_group = df_candidates.query(
                "mixing_group == @mix_group and majority_ko == @maj_ko"
            )

        if df_group.empty:
            print(f"  -> mouse {mouse_id}, {label}: 0 candidates, skipping.")
            continue

        df_sel = farthest_point_sampling(df_group, target_n, rng)
        df_sel = df_sel.copy()
        df_sel["sample_group"] = label

        records.append(df_sel)

        print(
            f"  -> mouse {mouse_id}, {label}: "
            f"selected {df_sel.shape[0]} ROIs (target {target_n})"
        )

# concatenate all selected ROIs
if records:
    roi_df = pd.concat(records, ignore_index=True)
else:
    roi_df = pd.DataFrame()

print("ROI table shape:", roi_df.shape)
display(roi_df.head())


# ============================================
# Store in adata.uns and write CSVs
# ============================================
if roi_df.shape[0] == 0:
    print("No ROIs found – nothing to write.")
else:
    A.uns["ko_rois"] = {
        "config": {
            "KO_COL": KO_COL,
            "KO_WHITELIST": list(KO_WHITELIST),
            "MOUSE_COL": MOUSE_COL,
            "SPATIAL_KEY": SPATIAL_KEY,
            "ROI_RADIUS": ROI_RADIUS,
            "ROI_AREA": ROI_AREA,
            "MIN_CELLS_ROI": MIN_CELLS_ROI,
            "MIN_KO_CELLS": MIN_KO_CELLS,
            "MIX_THRESHOLD": MIX_THRESHOLD,
            "N_LOW_F8": N_LOW_F8,
            "N_LOW_SE1": N_LOW_SE1,
            "N_HIGH_MIX": N_HIGH_MIX,
            "ALPHA_TUMOR": ALPHA_TUMOR,
            "TUMOR_BORDER_WIDTH": TUMOR_BORDER_WIDTH,
            "MIN_TUMOR_AREA_FRAC": MIN_TUMOR_AREA_FRAC,
        },
        "phenotype_levels": phenotype_levels,
        "table": roi_df.to_dict(orient="list"),
    }

    out_dir = "../analysis/TME_ROIs/"
    os.makedirs(out_dir, exist_ok=True)
    
    csv_all      = os.path.join(out_dir, "ko_ROIs_infiltration_SE1_F8_all.csv")
    csv_low_f8   = os.path.join(out_dir, "ko_ROIs_low_mixing_F8_SE1_F8.csv")
    csv_low_se1  = os.path.join(out_dir, "ko_ROIs_low_mixing_Serpine1_SE1_F8.csv")
    csv_high_any = os.path.join(out_dir, "ko_ROIs_high_mixing_any_SE1_F8.csv")

    roi_df.to_csv(csv_all, index=False)
    roi_df.query("sample_group == 'low_F8'").to_csv(csv_low_f8, index=False)
    roi_df.query("sample_group == 'low_Serpine1'").to_csv(csv_low_se1, index=False)
    roi_df.query("sample_group == 'high_mixing'").to_csv(csv_high_any, index=False)

    print("Saved ROI CSVs:")
    print("  ALL              ->", csv_all)
    print("  LOW F8           ->", csv_low_f8)
    print("  LOW Serpine1     ->", csv_low_se1)
    print("  HIGH mixing ANY  ->", csv_high_any)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

# Same EXCLUDE_BOXES dict as in the ROI script

def plot_rois_for_mouse(mouse_id, roi_df, coords, mice,
                        color_by="sample_group",
                        show_exclude_boxes=True,
                        alpha_cells=0.2,
                        s_cells=1.0,
                        figsize=(6, 6),
                        save_path=None):
    """
    Visualise ROIs over the tissue for a given mouse.
    
    Parameters
    ----------
    mouse_id : str
        Mouse/library ID to plot.
    roi_df : pd.DataFrame
        ROI table from the sampling script.
    coords : np.ndarray
        All cell coordinates (N_cells, 2) from adata.obsm[SPATIAL_KEY].
    mice : np.ndarray
        Mouse IDs per cell (same length as coords).
    color_by : {"mixing_group", "majority_ko", "sample_group"}
        Which column in roi_df to use for ROI colours.
        - mixing_group: "low_mixing", "high_mixing"
        - majority_ko : e.g. "F8", "Serpinb2" or "Serpine1"
        - sample_group: e.g. "low_F8", "low_Serpinb2"/"low_Serpine1", "high_mixing"
    show_exclude_boxes : bool
        If True, draw any EXCLUDE_BOXES[mouse_id] as rectangles.
    save_path : str or None
        If given, save the figure instead of (or in addition to) showing.
    """

    # Subset cells of this mouse
    mask_mouse = (mice == mouse_id)
    coords_mouse = coords[mask_mouse, :]

    if coords_mouse.shape[0] == 0:
        print(f"[WARN] No cells for mouse {mouse_id}")
        return

    # Subset ROIs of this mouse
    df_mouse = roi_df.query("mouse == @mouse_id")
    if df_mouse.shape[0] == 0:
        print(f"[WARN] No ROIs for mouse {mouse_id}")
        return

    # -----------------------------
    # Choose colours for groups
    # -----------------------------
    if color_by == "mixing_group":
        # low vs high mixing
        color_map = {
            "low_mixing":  "tab:blue",
            "high_mixing": "tab:red",
        }

    elif color_by == "majority_ko":
        # Handle F8 + Serpine1 OR F8 + Serpinb2
        color_map = {}
        if "majority_ko" in df_mouse.columns:
            unique_kos = sorted(df_mouse["majority_ko"].dropna().unique())
            for ko in unique_kos:
                if ko == "F8":
                    color_map[ko] = "tab:orange"
                else:
                    # whatever the other KO is (Serpinb2 or Serpine1)
                    color_map[ko] = "tab:green"
        else:
            color_map = {}

    elif color_by == "sample_group":
        # Groups from the new scripts:
        #   - low_F8
        #   - low_Serpinb2 OR low_Serpine1
        #   - high_mixing
        base_sample_colors = {
            "low_F8":        "tab:blue",
            "low_Serpinb2":  "tab:green",
            "low_Serpine1":  "tab:green",
            "high_mixing":   "tab:red",
        }
        color_map = {}
        if "sample_group" in df_mouse.columns:
            unique_groups = sorted(df_mouse["sample_group"].dropna().unique())
            for g in unique_groups:
                color_map[g] = base_sample_colors.get(g, "tab:gray")
        else:
            color_map = {}

    else:
        # fallback: one colour (all ROIs same)
        color_map = {}

    fig, ax = plt.subplots(figsize=figsize)

    # Plot all cells in the background
    ax.scatter(coords_mouse[:, 0], coords_mouse[:, 1],
               s=s_cells, alpha=alpha_cells, color="lightgrey")

    # Draw ROIs as circles
    for _, row in df_mouse.iterrows():
        cx = row["center_x"]
        cy = row["center_y"]
        radius = row["roi_radius"]

        if (color_by in row) and (row[color_by] in color_map):
            col = color_map[row[color_by]]
        else:
            col = "tab:gray"

        circ = Circle((cx, cy), radius,
                      fill=False, ec=col, lw=0.7, alpha=0.9)
        ax.add_patch(circ)

    # Optional: show exclusion boxes used in sampling
    if show_exclude_boxes and 'EXCLUDE_BOXES' in globals():
        boxes = EXCLUDE_BOXES.get(mouse_id, [])
        for (xmin, xmax, ymin, ymax) in boxes:
            rect = Rectangle((xmin, ymin),
                             width=xmax - xmin,
                             height=ymax - ymin,
                             fill=False,
                             ec="black",
                             lw=1.0,
                             linestyle="--",
                             alpha=0.8)
            ax.add_patch(rect)

    ax.set_title(f"ROIs for mouse {mouse_id} (color by {color_by})")
    ax.set_aspect("equal")
    # ax.invert_yaxis()  # optional

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight", dpi=300)
        print("Saved:", save_path)
        plt.close(fig)
    else:
        plt.show()
